# 03 — Config Driven Pipeline

Purpose: keep pipeline rules in a config instead of hardcoding them directly in transformation logic.

Process schema:

```text
BRONZE INPUT
      |
      v
CONFIG
- allowed_event_types
- min_amount
      |
      v
TRANSFORMS
      |
      +--> SILVER
      |
      +--> QUARANTINE
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("config-driven-pipeline")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_patterns_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Pipeline config

In [2]:
config = {
    "allowed_event_types": ["purchase", "refund"],
    "min_amount": 0.0,
}

config

{'allowed_event_types': ['purchase', 'refund'], 'min_amount': 0.0}

## Step 2 — Bronze input

In [3]:
bronze_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time_raw", StringType(), True),
])

rows = [
    ("e1", "u1", "Purchase", 10.0, "2026-01-01 10:00:00"),
    ("e2", None, "purchase", 20.0, "2026-01-01 10:05:00"),
    ("e3", "u2", "refund", 5.0, "2026-01-01 10:10:00"),
    ("e4", "u3", "unknown", 7.0, "2026-01-01 10:15:00"),
    ("e5", "u4", "purchase", -1.0, "2026-01-01 10:20:00"),
    ("e6", "u5", "refund", None, "bad-date"),
]

bronze = spark.createDataFrame(rows, bronze_schema)

bronze.show(truncate=False)
bronze.printSchema()

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|event_time_raw     |
+--------+-------+----------+------+-------------------+
|e1      |u1     |Purchase  |10.0  |2026-01-01 10:00:00|
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|
|e6      |u5     |refund    |NULL  |bad-date           |
+--------+-------+----------+------+-------------------+

root
 |-- event_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- event_time_raw: string (nullable = true)



## Step 3 — Transform functions using config

In [4]:
def normalize(df):
    return (
        df
        .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
        .withColumn("user_id", F.trim(F.col("user_id")))
    )


def parse_event_time(df):
    return df.withColumn("event_time", F.to_timestamp("event_time_raw"))


def apply_config_rules(df, cfg):
    return (
        df
        .withColumn(
            "error_reason",
            F.when(F.col("event_id").isNull(), F.lit("missing_event_id"))
             .when(F.col("user_id").isNull() | (F.col("user_id") == ""), F.lit("missing_user_id"))
             .when(~F.col("event_type").isin(cfg["allowed_event_types"]), F.lit("invalid_event_type"))
             .when(F.col("amount").isNull(), F.lit("missing_amount"))
             .when(F.col("amount") < F.lit(cfg["min_amount"]), F.lit("invalid_amount"))
             .when(F.col("event_time").isNull(), F.lit("invalid_event_time"))
        )
        .withColumn("is_valid", F.col("error_reason").isNull())
    )

## Step 4 — Run pipeline

In [5]:
prepared = (
    bronze
    .transform(normalize)
    .transform(parse_event_time)
    .transform(lambda df: apply_config_rules(df, config))
)

prepared.show(truncate=False)

[Stage 2:>                                                          (0 + 1) / 1]

+--------+-------+----------+------+-------------------+-------------------+------------------+--------+
|event_id|user_id|event_type|amount|event_time_raw     |event_time         |error_reason      |is_valid|
+--------+-------+----------+------+-------------------+-------------------+------------------+--------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|2026-01-01 10:00:00|NULL              |true    |
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|2026-01-01 10:05:00|missing_user_id   |false   |
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|2026-01-01 10:10:00|NULL              |true    |
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|2026-01-01 10:15:00|invalid_event_type|false   |
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|2026-01-01 10:20:00|invalid_amount    |false   |
|e6      |u5     |refund    |NULL  |bad-date           |NULL               |missing_amount    |false   |
+--------+-------+----------+------+-------------------

## Step 5 — Silver output

In [6]:
silver = (
    prepared
    .filter(F.col("is_valid"))
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "event_time",
    )
)

silver.show(truncate=False)

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|event_time         |
+--------+-------+----------+------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-01-01 10:00:00|
|e3      |u2     |refund    |5.0   |2026-01-01 10:10:00|
+--------+-------+----------+------+-------------------+



## Step 6 — Quarantine output

In [7]:
quarantine = (
    prepared
    .filter(~F.col("is_valid"))
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "event_time_raw",
        "error_reason",
    )
)

quarantine.show(truncate=False)

+--------+-------+----------+------+-------------------+------------------+
|event_id|user_id|event_type|amount|event_time_raw     |error_reason      |
+--------+-------+----------+------+-------------------+------------------+
|e2      |NULL   |purchase  |20.0  |2026-01-01 10:05:00|missing_user_id   |
|e4      |u3     |unknown   |7.0   |2026-01-01 10:15:00|invalid_event_type|
|e5      |u4     |purchase  |-1.0  |2026-01-01 10:20:00|invalid_amount    |
|e6      |u5     |refund    |NULL  |bad-date           |missing_amount    |
+--------+-------+----------+------+-------------------+------------------+



## Step 7 — Change config and rerun

Example: accept only purchases.

In [8]:
purchase_only_config = {
    "allowed_event_types": ["purchase"],
    "min_amount": 0.0,
}

prepared_purchase_only = (
    bronze
    .transform(normalize)
    .transform(parse_event_time)
    .transform(lambda df: apply_config_rules(df, purchase_only_config))
)

prepared_purchase_only.select(
    "event_id",
    "event_type",
    "amount",
    "is_valid",
    "error_reason"
).show(truncate=False)

+--------+----------+------+--------+------------------+
|event_id|event_type|amount|is_valid|error_reason      |
+--------+----------+------+--------+------------------+
|e1      |purchase  |10.0  |true    |NULL              |
|e2      |purchase  |20.0  |false   |missing_user_id   |
|e3      |refund    |5.0   |false   |invalid_event_type|
|e4      |unknown   |7.0   |false   |invalid_event_type|
|e5      |purchase  |-1.0  |false   |invalid_amount    |
|e6      |refund    |NULL  |false   |invalid_event_type|
+--------+----------+------+--------+------------------+



## Step 8 — Control totals

In [9]:
control_totals_schema = StructType([
    StructField("metric", StringType(), False),
    StructField("value", LongType(), False),
])

control_totals = spark.createDataFrame(
    [
        ("bronze_rows", bronze.count()),
        ("prepared_rows", prepared.count()),
        ("silver_rows", silver.count()),
        ("quarantine_rows", quarantine.count()),
    ],
    control_totals_schema
)

control_totals.show(truncate=False)

+---------------+-----+
|metric         |value|
+---------------+-----+
|bronze_rows    |6    |
|prepared_rows  |6    |
|silver_rows    |2    |
|quarantine_rows|4    |
+---------------+-----+



## Final result

```text
BRONZE
  |
  v
CONFIG-DRIVEN RULES
  |
  +--> SILVER
  |
  +--> QUARANTINE
```

In [10]:
spark.stop()